In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick
import matplotlib.colors as mcolors
import os
import glob
import json
import math

sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.autolayout': True, 'figure.dpi': 120})

In [ ]:
def load_benchmark_data(data_dir="./datasets"):
    """
    Scans the data directory for pyperf JSON files, extracting all individual
    benchmark iterations into a long-form Pandas DataFrame.
    """
    records = []
    
    # Mapping dictionaries for prettier plot labels
    phase_map = {
        "soa": "AoS to SoA",
        "pp": "K-Means++ Init",
        "lloyd": "Lloyd Iterations"
    }
    lang_map = {
        "cpp": "C++ (EVE)",
        "py": "Python (Scikit-Learn)"
    }
    
    json_files = glob.glob(os.path.join(data_dir, "*.json"))
    
    for filepath in json_files:
        filename = os.path.basename(filepath)
        name_no_ext = os.path.splitext(filename)[0]
        
        # Expected format: {phase}_{lang}_{dim}D_{samples}S_{clusters}K
        try:
            parts = name_no_ext.split("_")
            phase_key = parts[0]
            lang_key = parts[1]
            dim = int(parts[2].replace("D", ""))
            samples = int(parts[3].replace("S", ""))
            clusters = int(parts[4].replace("K", ""))
        except (IndexError, ValueError):
            print(f"Skipping malformed file: {filename}")
            continue
            
        with open(filepath, 'r') as f:
            data = json.load(f)
            
        # Parse the standard Pyperf JSON schema
        for bench in data.get("benchmarks", []):
            for run in bench.get("runs", []):
                # 'values' contains the array of actual iterations (epochs)
                values = run.get("values", [])
                for val in values:
                    records.append({
                        "Phase": phase_map.get(phase_key, phase_key),
                        "Language": lang_map.get(lang_key, lang_key),
                        "Dimensions": dim,
                        "Samples": samples,
                        "Clusters": clusters,
                        "Time_s": val,
                        "Time_ms": val * 1000.0,
                        "Configuration": f"{dim}D | {samples}S | {clusters}K"
                    })
                    
    df = pd.DataFrame(records)
    
    # Convert categorical columns for proper sorting in Seaborn plots
    df["Phase"] = pd.Categorical(df["Phase"], categories=["AoS to SoA", "K-Means++ Init", "Lloyd Iterations"], ordered=True)
    df["Language"] = pd.Categorical(df["Language"], categories=["C++ (EVE)", "Python (Scikit-Learn)"], ordered=True)
    
    return df

df_bench = load_benchmark_data("./datasets")

print(f"Loaded {len(df_bench)} individual iterations.")
df_bench.head()

In [ ]:
import glob
import os
import numpy as np
import pandas as pd

def compute_inertia(data_file, result_file, n_samples, n_features):
    """Reads the result file and calculates the sum of squared distances to centroids."""
    raw_data = np.fromfile(data_file, dtype=np.float32).reshape((n_samples, n_features))
    
    with open(result_file, 'r') as f:
        lines = f.read().splitlines()
        
    centroids = []
    clusters = {}
    
    mode = None
    for line in lines:
        if line == "[Centroids]":
            mode = "centroids"
            continue
        elif line == "[Clusters]":
            mode = "clusters"
            continue
            
        if mode == "centroids":
            centroids.append([float(x) for x in line.split()])
        elif mode == "clusters":
            parts = line.split(":")
            k = int(parts[0])
            indices_str = parts[1].strip()
            clusters[k] = [int(x) for x in indices_str.split()] if indices_str else []
            
    centroids = np.array(centroids, dtype=np.float32)
    inertia = 0.0
    
    for k, indices in clusters.items():
        if not indices:
            continue
        pts = raw_data[indices]
        c = centroids[k]
        inertia += np.sum((pts - c) ** 2)
        
    return inertia

def validate_lloyd_parity(data_dir="./datasets", tolerance_pct=0.01):
    """
    Sweeps the directory for all C++ results, finds the matching Python result,
    and asserts that their inertias match within a floating-point tolerance.
    """
    cpp_files = glob.glob(os.path.join(data_dir, "results_cpp_*.txt"))
    records = []
    
    for cpp_file in cpp_files:
        filename = os.path.basename(cpp_file)
        # Reconstruct the config identifier and corresponding files
        config_id = filename.replace("results_cpp_", "").replace(".txt", "")
        py_file = os.path.join(data_dir, f"results_py_{config_id}.txt")
        data_file = os.path.join(data_dir, f"data_{config_id}.bin")
        
        if not os.path.exists(py_file) or not os.path.exists(data_file):
            print(f"Missing matching Python or Data file for {config_id}. Skipping.")
            continue
            
        # Extract dimensions and samples from the config_id (e.g., "3D_1000S_5K")
        parts = config_id.split("_")
        dim = int(parts[0].replace("D", ""))
        samples = int(parts[1].replace("S", ""))
        clusters = int(parts[2].replace("K", ""))
        
        # Compute both inertias
        cpp_inertia = compute_inertia(data_file, cpp_file, samples, dim)
        py_inertia = compute_inertia(data_file, py_file, samples, dim)
        
        # Calculate percentage difference
        max_inertia = max(cpp_inertia, py_inertia)
        diff_pct = abs(cpp_inertia - py_inertia) / max_inertia * 100 if max_inertia > 0 else 0.0
        
        status = "✅ PASS" if diff_pct <= tolerance_pct else "❌ FAIL"
        
        records.append({
            "Configuration": f"{dim}D | {samples}S | {clusters}K",
            "C++ Inertia": cpp_inertia,
            "Python Inertia": py_inertia,
            "Diff (%)": diff_pct,
            "Status": status
        })
        
    df_parity = pd.DataFrame(records)
    
    # Print a quick summary
    failures = df_parity[df_parity["Status"] == "❌ FAIL"]
    if not failures.empty:
        print(f"WARNING: Found {len(failures)} configurations outside the {tolerance_pct}% tolerance limit!")
    else:
        print(f"SUCCESS: All {len(df_parity)} configurations match within the {tolerance_pct}% tolerance limit!")
        
    # Sort by the largest differences to see where floating point math deviated the most
    return df_parity.sort_values(by="Diff (%)", ascending=False).reset_index(drop=True)

# Run the validation
df_parity = validate_lloyd_parity(tolerance_pct=0.01)
display(df_parity)

In [ ]:

largest_samples = df_bench["Samples"].max()
df_cpp = df_bench[
    (df_bench["Language"] == "C++ (EVE)") & 
    (df_bench["Samples"] == largest_samples)
]

unique_clusters = sorted(df_cpp["Clusters"].unique())
n_plots = len(unique_clusters)

cols = 2
rows = math.ceil(n_plots / cols)

fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(12, 5 * rows), sharey=True)

axes = axes.flatten() if n_plots > 1 else [axes]

colors = ["#e74c3c", "#f39c12", "#3498db"]

for i, cluster in enumerate(unique_clusters):
    ax = axes[i]
    
    df_cluster = df_cpp[df_cpp["Clusters"] == cluster]
    
    # Group, pivot, and calculate percentages
    df_grouped = df_cluster.groupby(["Dimensions", "Phase"], observed=True)["Time_s"].mean().reset_index()
    df_pivot = df_grouped.pivot(index="Dimensions", columns="Phase", values="Time_s")
    df_memory_pct = df_pivot.div(df_pivot.sum(axis=1), axis=0) * 100
    
    df_memory_pct.plot(
        kind='bar', 
        stacked=True, 
        ax=ax,
        color=colors,
        legend=False
    )
    
    ax.set_title(f"{cluster} Clusters")
    ax.set_xlabel("Dimensions")
    ax.tick_params(axis='x', rotation=0)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    if i % cols == 0:
        ax.set_ylabel("Percentage of Total Time (%)")
    else:
        ax.set_ylabel("")

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle(
    f"Normalized C++ Execution Breakdown: Memory vs Init vs Math\n(Fixed at {largest_samples:,} Samples)", 
    fontsize=16
)

# Add a single global legend using the handles from the first plot
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title="Execution Phase", bbox_to_anchor=(1.05, 0.95), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
df_math = df_bench[df_bench['Phase'] == 'Lloyd Iterations'].copy()
df_math['Iteration'] = df_math.groupby(['Language', 'Dimensions', 'Samples', 'Clusters']).cumcount()

df_pivot = df_math.pivot(
    index=['Dimensions', 'Samples', 'Clusters', 'Iteration'], 
    columns='Language', 
    values='Time_s'
).reset_index()

df_pivot['Speedup (x)'] = df_pivot['Python (Scikit-Learn)'] / df_pivot['C++ (EVE)']
df_pivot['Cluster Group'] = df_pivot['Clusters'].astype(str) + " Clusters"

g = sns.relplot(
    data=df_pivot,
    x="Samples",
    y="Speedup (x)",
    hue="Cluster Group",    
    col="Dimensions",       
    col_wrap=2,             
    kind="scatter",
    alpha=0.6,     
    palette="Set1",         
    height=4,
    aspect=1.2,
    facet_kws={'sharey': False} 
)

g.set(xscale="log")
g.fig.suptitle("Absolute Speedup vs. Sample Size (Lloyd Iterations)", y=1.05)
g.set_axis_labels("Number of Samples", "Speedup Multiplier (x)")

for ax in g.axes.flat:
    ax.grid(True, linestyle='--', alpha=0.7)

plt.show()

df_pivot.drop(columns=['Cluster Group', 'Iteration'], inplace=True, errors='ignore')

In [ ]:
df_lloyd = df_bench[df_bench["Phase"] == "Lloyd Iterations"]
df_mean = df_lloyd.groupby(["Dimensions", "Samples", "Clusters", "Language"], observed=True)["Time_s"].mean().reset_index()
df = df_mean.pivot(index=["Dimensions", "Samples", "Clusters"], columns="Language", values="Time_s").reset_index()
df["Speedup (x)"] = df["Python (Scikit-Learn)"] / df["C++ (EVE)"]

cluster_vals = sorted(df["Clusters"].unique())
n_clusters = len(cluster_vals)

n_cols = 2
n_rows = math.ceil(n_clusters / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 6 * n_rows))

if n_clusters > 1:
    axes = axes.flatten()
else:
    axes = [axes]

for i, cluster in enumerate(cluster_vals):
    ax = axes[i]
    
    subset = df[df["Clusters"] == cluster]
    
    heatmap_data = subset.groupby(["Dimensions", "Samples"])["Speedup (x)"].mean().reset_index()
    heatmap_pivot = heatmap_data.pivot(index="Dimensions", columns="Samples", values="Speedup (x)")
    
    sns.heatmap(
        heatmap_pivot, 
        annot=True,
        fmt=".2f", 
        cmap="turbo",      
        ax=ax,
        cbar=True,
        cbar_kws={'label': 'Speedup Multiplier (x)'},
        linewidths=.5
    )
    
    ax.set_title(f"Clusters: {cluster}")
    ax.set_xlabel("Number of Samples")
    
    if i % n_cols == 0:
        ax.set_ylabel("Dimensions")
    else:
        ax.set_ylabel("")
        
    ax.tick_params(axis='x', labelrotation=0)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Speedup Heatmap (C++ vs. Scikit-Learn) by Number of Clusters\nPhase: Lloyd Iterations", y=1.02, fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
py_means = df_bench[df_bench["Language"] == "Python (Scikit-Learn)"].groupby(
    ["Phase", "Dimensions", "Samples", "Clusters"], observed=True
)["Time_s"].mean().reset_index().rename(columns={"Time_s": "Py_Mean_Time"})

df_cpp = df_bench[df_bench["Language"] == "C++ (EVE)"].copy()
df_norm = pd.merge(df_cpp, py_means, on=["Phase", "Dimensions", "Samples", "Clusters"])
df_norm["Speedup (x)"] = df_norm["Py_Mean_Time"] / df_norm["Time_s"]

base_k = df_norm["Clusters"].min()
baseline_df = df_norm[df_norm["Clusters"] == base_k].groupby(
    ["Phase", "Dimensions", "Samples"], observed=True
)["Speedup (x)"].mean().reset_index().rename(columns={"Speedup (x)": "Base_Speedup"})

df_norm = pd.merge(df_norm, baseline_df, on=["Phase", "Dimensions", "Samples"])
df_norm["Performance Retention (%)"] = (df_norm["Speedup (x)"] / df_norm["Base_Speedup"]) * 100

df_norm["Phase"] = df_norm["Phase"].cat.remove_unused_categories()

for phase in df_norm["Phase"].unique():
    phase_data = df_norm[df_norm["Phase"] == phase]
    
    g = sns.relplot(
        data=phase_data,
        x="Clusters",
        y="Performance Retention (%)",
        hue="Samples",   
        col="Dimensions",
        col_wrap=2,
        kind="line",
        errorbar="sd",
        marker="o",
        hue_norm=mcolors.LogNorm(),
        palette="turbo",
        height=4,  
        aspect=1.2,
        facet_kws={'sharey': True}
    )

    # Styling the header to make the phase separation visually distinct
    g.fig.suptitle(
        f"PHASE: {phase.upper()}\nNormalized Efficiency against K={base_k}", 
        y=1.08, 
        fontsize=16, 
        fontweight='bold',
        bbox={'facecolor': 'lightgrey', 'alpha': 0.3, 'pad': 10}
    )
    
    g.set_axis_labels("Number of Clusters (K)", "Retention (%)")

    for ax in g.axes.flat:
        ax.axhline(100, color='red', linestyle='--', alpha=0.5)
        ax.grid(axis='y', linestyle='--', alpha=0.7)
        ax.xaxis.set_major_locator(mtick.MaxNLocator(integer=True))
        
    plt.show()